# HTS-AT: transfer learning com encoder congelado

Este notebook segue o fluxo do AST e treina somente uma nova camada linear para 13 efeitos. Fine-tuning do encoder não faz parte desta etapa.

**Estado inicial: preparação concluída, treinamento e extração completa desativados.** As duas etapas longas exigem controles explícitos abaixo. Os outputs históricos do AST não são resultados deste experimento.

Leia `docs/transfer_learning_consolidation.md` e escolha o kernel **TCC Transfer Learning (PyTorch CUDA)**.

## 1. Importações e raiz do projeto

In [ ]:
from pathlib import Path
import sys
import json
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Abra o notebook a partir da raiz do TCC ou de notebooks/.")
sys.path.insert(0, str(ROOT))

from src.training.linear_probe import (load_config, preflight, extract_embeddings,
    validate_cache, train_probe, paths_for, device_for)
from src.models.transfer import load_frozen_encoder, FrozenProbe
from src.data.transfer import read_manifest, manifest_path, EFFECTS


## 2. Configuração

O primeiro cenário é Mono Discrete. Os dois modelos usam os mesmos WAVs e as mesmas partições por áudio original: aproximadamente 72/8/20, seed 42. MT2 e todas as pastas NoFX ficam excluídos.

As features são calculadas pelo frontend oficial de cada checkpoint: 128 bandas para PaSST, 64 para HTS-AT. Os ZIPs `mel32` do colega ficam preservados e não são usados como se reproduzissem esses frontends. Os áudios reais de 2 segundos são preservados; não há padding automático a 10 segundos copiado do AST.

In [ ]:
MODEL = "htsat"
SCENARIO = "mono_disc"
cfg = load_config(ROOT / "configs/linear_probe" / f"{MODEL}_mono_disc.json")
cfg["scenario"] = SCENARIO

# Controles de execução: permanecem False nesta entrega.
RUN_EXTRACTION = False
RUN_TRAINING = False

print(json.dumps(cfg, indent=2))
print("Extração habilitada:", RUN_EXTRACTION, "Treinamento habilitado:", RUN_TRAINING)


## 3. Dados e verificação prévia

O índice local registra classe, áudio de origem, split e SHA-256 de cada WAV. O loader falha se um áudio muda; não descarta amostras silenciosamente.

Para restaurar outro cenário, use primeiro `scripts/prepare_transfer_data.py --scenario ... --extract` e depois `scripts/prepare_transfer_experiment.py index --scenario ...`. Não execute a restauração dentro do notebook.

In [ ]:
status = preflight(cfg)
print(json.dumps(status, indent=2))
if not status["ready"]:
    raise RuntimeError("Preparação incompleta: " + "; ".join(status["errors"]))
rows = read_manifest(SCENARIO)
print("Índice:", manifest_path(SCENARIO))
print("Classes:", list(EFFECTS))


## 4. Modelo pré-treinado e cabeça nova

Esta célula apenas carrega os pesos e cria uma cabeça não treinada. O encoder fica congelado e em modo `eval`, inclusive quando a cabeça passa para `train`. Não há download automático nem substituição silenciosa por pesos aleatórios. Para classificar um único efeito por áudio, usamos logits de 13 classes e CrossEntropyLoss.

No HTS-AT, usamos o vetor latente anterior à cabeça AudioSet; este é um experimento de linear probing, não ajuste da camada token-semantic nem detecção temporal.

In [ ]:
device = device_for(cfg)
encoder = load_frozen_encoder(MODEL)
probe = FrozenProbe(encoder, classes=len(EFFECTS)).to(device)
probe.train()
assert not probe.encoder.training
assert all(not p.requires_grad for p in probe.encoder.parameters())
print("Device:", device)
print("Parâmetros treináveis:", sum(p.numel() for p in probe.parameters() if p.requires_grad))
print("Encoder congelado. Nenhuma atualização de pesos realizada.")
# A etapa de extração carrega seu próprio encoder; evite manter uma cópia na GPU.
del probe, encoder
if device.type == "cuda":
    torch.cuda.empty_cache()


## 5. Extração de embeddings — etapa futura explícita

Quando decidir iniciar, habilite `RUN_EXTRACTION` e execute a configuração e esta célula. Uma passagem de inferência gera 768 valores por áudio, com autocast bfloat16 no encoder e frontend float32. Isso permite reutilizar os embeddings nas 10 épocas da cabeça, sem executar novamente o Transformer em cada época.

O cache local contém fingerprint do índice, checkpoint, versões e implementação. Um cache incompatível é recusado. A extração não usa rótulos para adaptar o encoder, e pode incluir as três partições; o teste não participa do treinamento da cabeça. Cerca de 0,35 GiB de embeddings são necessários para 123.552 amostras.

Nenhum cache de embeddings deste cenário foi produzido na preparação.

In [ ]:
if RUN_EXTRACTION:
    cache = extract_embeddings(cfg, allow_extraction=True)
    print("Cache:", cache)
else:
    print("Extração completa desativada. Nenhum embedding de treinamento foi gerado.")


## 6. Linear probing — não executar ainda

Ao habilitar `RUN_TRAINING`, a rotina exige o cache completo e válido, treina somente a cabeça, escolhe o melhor checkpoint pela acurácia de validação e então avalia esse checkpoint no teste.

Default: 10 épocas, AdamW, LR 0,001, batch de embeddings 1024. O batch de extração é 16. A cabeça recebe embeddings de 768 dimensões; seus batches têm custo muito menor que batches de espectrogramas. Cada execução cria sua própria pasta de resultados. Não há função para desbloquear o encoder.

In [ ]:
if RUN_TRAINING:
    run_dir, metrics = train_probe(cfg, allow_training=True)
    print("Resultados:", run_dir)
    print(json.dumps(metrics, indent=2))
else:
    print("Treinamento desativado. Nenhuma época ou optimizer.step executado.")


## 7. Resultados e rastreabilidade

Uma execução futura salva `best_head.pt`, `history.json`, `test_metrics.json`, `confusion_matrix.png` e `run.json`, incluindo classes, configuração e identidade do cache. Os arquivos ficam em `results/transfer_learning/<cenario>/<modelo>/` e são ignorados pelo Git.

A estimativa da primeira execução considera extração + 10 épocas da cabeça; o relatório em `data/audits/transfer_learning/` identifica o hardware, as medições curtas e suas limitações. Nenhum fine-tuning está previsto aqui.

In [ ]:
folder, _, _ = paths_for(cfg)
print("Saídas futuras:", folder)
print("Preparação concluída; para iniciar, habilite conscientemente as etapas 5 e 6.")
